In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity


In [7]:
movies = pd.read_csv(
    "movies.dat",
    sep="::",
    engine="python",
    names=["movie_id", "title", "genres"],
    encoding="latin-1"
)

movies.head()


,movie_id,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


In [8]:
movies['genres_list'] = movies['genres'].str.split('|')
movies[['title', 'genres_list']].head()


,title,genres_list
0,Toy Story (1995),"[Animation, Children's, Comedy]"
1,Jumanji (1995),"[Adventure, Children's, Fantasy]"
2,Grumpier Old Men (1995),"[Comedy, Romance]"
3,Waiting to Exhale (1995),"[Comedy, Drama]"
4,Father of the Bride Part II (1995),[Comedy]


In [9]:
#convert the genre into numeric vectors

from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(movies['genres_list'])

genre_df = pd.DataFrame(
    genre_matrix,
    columns=mlb.classes_,
    index=movies.index
)

genre_df.head()


,Action,Adventure,Animation,Children's,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0
2,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0
3,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0


In [10]:
#compare movie based on genre vectors

from sklearn.metrics.pairwise import cosine_similarity

genre_similarity = cosine_similarity(genre_df)


In [11]:
def recommend_by_genre(movie_title, movies, similarity_matrix, top_n=5):

    if movie_title not in movies['title'].values:
        return "Movie not found."

    # index of the movie
    idx = movies[movies['title'] == movie_title].index[0]

    # similarity scores
    sim_scores = list(enumerate(similarity_matrix[idx]))

    # sort by similarity
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # skip the movie itself
    top_indices = [i[0] for i in sim_scores[1:top_n + 1]]

    return movies.iloc[top_indices][['title', 'genres']]


In [12]:
#test

recommend_by_genre(
    "Toy Story (1995)",
    movies,
    genre_similarity,
    top_n=5
)


,title,genres
1050,Aladdin and the King of Thieves (1996),Animation|Children's|Comedy
2072,"American Tail, An (1986)",Animation|Children's|Comedy
2073,"American Tail: Fievel Goes West, An (1991)",Animation|Children's|Comedy
2285,"Rugrats Movie, The (1998)",Animation|Children's|Comedy
2286,"Bug's Life, A (1998)",Animation|Children's|Comedy
